In [1]:
!pip install gymnasium torch numpy

In [2]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import copy
from collections import deque


class ReplayBuffer:
    def __init__(self, max_size):
        self.buffer = deque(maxlen=max_size)

    def store(self, state, action, reward, next_state, done):
        experience = (state, action, reward, next_state, done)
        self.buffer.append(experience)

    def sample(self, batch_size):
        if len(self.buffer) >= batch_size:
            return random.sample(self.buffer, k=batch_size)
        else:
            return None



class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.network(x)



class Agent:
    def __init__(self, gamma, epsilon, epsilon_decay, batch_size, target_update):
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update = target_update

        self.policy_network = Model()
        self.buffer = ReplayBuffer(50000)
        self.optimizer = optim.Adam(self.policy_network.parameters(), lr=0.001)
        self.target_network = copy.deepcopy(self.policy_network)

    # Generate a random number. If it's less than epsilon take a random value, if not then argmax
    def choose_action(self, state):
        choice = random.random()
        if choice < self.epsilon:
            return random.randint(0, 1)
        else:
            state = torch.tensor(state, dtype=torch.float32)
            q_values = self.policy_network(state)
            return q_values.argmax().item()

    # Loss computation and learning function for the policy network
    def learn(self):
        batch = self.buffer.sample(self.batch_size)
        if batch is None:
            return
        prediction_list = []
        target_list = []

        for experience in batch:
            state, action, reward, next_state, done = experience

            state = torch.tensor(state, dtype=torch.float32)
            q_val = self.policy_network(state)
            prediction = q_val[action]
            prediction_list.append(prediction)

            next_state = torch.tensor(next_state, dtype=torch.float32)
            future_val = self.target_network(next_state)
            next_q_value = future_val.max().detach()

            if done:
                target = reward
            else:
                target = reward + self.gamma * next_q_value
            target_list.append(target)

        # Convert list to tensors for the MSELoss
        predictions = torch.stack(prediction_list)
        targets = torch.tensor(target_list)
        loss_fn = nn.MSELoss()
        loss = loss_fn(predictions, targets)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Epsilon to decrease exploration rate over time
        self.epsilon = max(self.epsilon * self.epsilon_decay, 0.01)

    # Update target network every N steps (called in training)
    def update_target_network(self):
        self.target_network.load_state_dict(self.policy_network.state_dict())



env = gym.make("CartPole-v1")
agent = Agent(gamma=0.99, epsilon=1.0, epsilon_decay=0.999, batch_size=64, target_update=1000)
step_counter = 0
max_episodes = 2000

for episode in range(max_episodes):
    state, info = env.reset()
    terminated = False
    truncated = False
    episode_steps = 0

    while not terminated and not truncated:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        agent.buffer.store(state, action, reward, next_state, done)
        state = next_state

        if step_counter % 4 == 0:
            agent.learn()

        if step_counter >= agent.target_update:
            step_counter = 0
            agent.update_target_network()
        step_counter += 1
        episode_steps += 1

    if episode % 50 == 0:
        print(f"Episode {episode}, Steps: {episode_steps}, Epsilon: {agent.epsilon:.3f}")

print("Training complete")

Episode 0, Steps: 12, Epsilon: 1.000
Episode 50, Steps: 14, Epsilon: 0.781
Episode 100, Steps: 17, Epsilon: 0.642
Episode 150, Steps: 15, Epsilon: 0.494
Episode 200, Steps: 215, Epsilon: 0.118
Episode 250, Steps: 214, Epsilon: 0.010
Episode 300, Steps: 305, Epsilon: 0.010
Episode 350, Steps: 222, Epsilon: 0.010
Episode 400, Steps: 142, Epsilon: 0.010
Episode 450, Steps: 184, Epsilon: 0.010
Episode 500, Steps: 135, Epsilon: 0.010
Episode 550, Steps: 174, Epsilon: 0.010
Episode 600, Steps: 500, Epsilon: 0.010
Episode 650, Steps: 405, Epsilon: 0.010
Episode 700, Steps: 192, Epsilon: 0.010
Episode 750, Steps: 411, Epsilon: 0.010
Episode 800, Steps: 500, Epsilon: 0.010
Episode 850, Steps: 410, Epsilon: 0.010
Episode 900, Steps: 500, Epsilon: 0.010
Episode 950, Steps: 407, Epsilon: 0.010
Episode 1000, Steps: 244, Epsilon: 0.010
Episode 1050, Steps: 500, Epsilon: 0.010
Episode 1100, Steps: 500, Epsilon: 0.010
Episode 1150, Steps: 302, Epsilon: 0.010
Episode 1200, Steps: 500, Epsilon: 0.010
Ep

In [3]:
torch.save(agent.policy_network.state_dict(), "cartpole_dqn.pth")